## Installing necessary libraries

Cohere provides free trial keys to use their LLMs. So generate one trial key from dashboard.cohere.com

In [4]:
#!pip install langchain-cohere langchain pdfminer.six chromadb

langchain-cohere: Enables integration of Cohere's language models with LangChain for advanced text generation and processing workflows.

langchain: Provides a modular framework for building language model-powered applications, such as chatbots, question-answering systems, and conversational agents.

pdfminer.six: Facilitates text extraction from PDF files, making it useful for document analysis and preprocessing tasks.

chromadb: A vector database library designed for efficient storage and retrieval of embeddings, ideal for tasks like semantic search and recommendation systems.

## Importing libraries

In [6]:
#!pip install -q --upgrade \
    langchain-core==0.3.28 \
    langchain-cohere==0.3.3 \
    langchain-community==0.3.12 \
    langchain-text-splitters==0.3.3 \
    langchain==0.3.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.9/326.9 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.4 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.28 which is incompatible.
langchain-classic 1.0.4 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.3 which is incompatible.
langgraph-p

In [7]:
#import importlib, site
#importlib.invalidate_caches()

In [1]:
import os
from google.colab import userdata
os.environ["COHERE_API_KEY"] = userdata.get('cohere_api')

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser        # ✅ moved here
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.documents import Document                    # ✅ moved here
from pydantic import BaseModel, Field
from langchain_cohere import ChatCohere
from langchain_community.vectorstores import Chroma              # ✅ moved here
from langchain_community.embeddings import CohereEmbeddings      # ✅ moved here
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ✅ moved here
from pdfminer.high_level import extract_text as extract_text_pdf_miner

## VectorDB setup

In [2]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/chroma_db"

# Initialize Cohere embeddings with the specified model
# "embed-english-v3.0" is a pre-trained English language embedding model by Cohere
# The user_agent parameter specifies the tool or library using the Cohere API, in this case, LangChain
embedding = CohereEmbeddings(
    model="embed-english-v3.0",
    user_agent="langchain"
)


/tmp/ipykernel_21364/2670508498.py:7: LangChainDeprecationWarning: The class `CohereEmbeddings` was deprecated in LangChain 0.0.30 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-cohere package and should be used instead. To use it run `pip install -U :class:`~langchain-cohere` and import as `from :class:`~langchain_cohere import CohereEmbeddings``.
  embedding = CohereEmbeddings(


We are processing 2 research papers on transformers and yolo. You can use your own PDFs.

In [3]:
# 1706.03762 = "Attention Is All You Need" (the Transformer paper)
!wget -q -O /content/1706.03762v7.pdf "https://arxiv.org/pdf/1706.03762v7"

# 1506.02640 = YOLO (You Only Look Once) object detection paper
!wget -q -O /content/1506.02640v5.pdf "https://arxiv.org/pdf/1506.02640v5"

print("Both PDFs downloaded!")

Both PDFs downloaded!


In [4]:
import os
for f in ["/content/1706.03762v7.pdf", "/content/1506.02640v5.pdf"]:
    size = os.path.getsize(f)
    print(f"{f} — {round(size/1024)} KB")

/content/1706.03762v7.pdf — 2163 KB
/content/1506.02640v5.pdf — 5173 KB


In [5]:
# Loop through a list of PDF files to process
for pdf_name in ["/content/1706.03762v7.pdf", "/content/1506.02640v5.pdf"]:
    # Open each PDF file in binary mode
    with open(pdf_name, 'rb') as f:
        # Extract text from the PDF using the extract_text_pdf_miner function
        text = extract_text_pdf_miner(f)

        # Clean the extracted text by removing newline characters and joining into a single string
        cleaned_text = " ".join(text.split("\n"))

        # Initialize a list to store document chunks
        docs = []

        # Create a text splitter to divide the text into manageable chunks
        # Each chunk has a maximum size of 2048 characters with a 512-character overlap
        splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

        # Split the cleaned text into chunks and wrap each chunk in a Document object
        for chunk in splitter.split_text(cleaned_text):
            docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))

    # Create a Chroma collection from the processed documents
    # Use the specified persist directory and embedding model for storage and retrieval
    vector_collection_fixed_size = Chroma.from_documents(
        documents=docs,
        persist_directory=persist_directory,
        embedding=embedding
    )

In [6]:
# Initialize a Chroma vector database
# The persist_directory specifies the location where the database is stored
# The embedding_function parameter provides the embedding model used for vector representation
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

/tmp/ipykernel_21364/1816146847.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)


In [7]:
# Perform a similarity search on the vector database
# The query "What is YOLO?" is used to find the most relevant documents
# k=1 specifies that the top 1 most similar document should be retrieved
# The method also returns relevance scores indicating how closely each document matches the query
vectordb.similarity_search_with_relevance_scores("What is YOLO?", k=1)

[(Document(metadata={'source': '/content/1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In real-world applications it is hard to predict all possible use cases and  YOLO is a fast, accurate object detector, making it ideal for computer vision applications. We connect YOLO to a webcam and verify that it maintains real-time performance,  \x0cVOC 2007 AP 59.2 54.2 43.2 36.5 -  Picasso AP Best F1 0.590 53.3 0.226 10.4 0.458 37.8 0.271 17.8 0.051 1.9  People-Art AP 45 26 32  YOLO R-CNN DPM Poselets [2] D&T [4]  (a) Picasso Dataset precision-recall curves.  (b) Quantitative results on the VOC 2007, Picasso, and People-Art Datasets. The Picasso Dataset evaluates on both AP and best F1 score.  Figure 5: Generalization results on Picasso and People-Art datasets.  Figure 6: Qualitative Results. YOLO running on sample artwork and natural images from the internet. It is mostly accurate a

## RAG pipeline

In [10]:
# Initialize an LLM instance using Cohere's "command-r" model
# The temperature parameter controls randomness in the generated responses; 0 ensures deterministic outputs
llm = ChatCohere(model="command-r-plus-08-2024", temperature=0)

# Define a prompt template for generating answers based on a given context and question
prompt_str = """Answer the question below using the context:

Context: {context}

Question: {question}

Answer: """

# Create a ChatPromptTemplate from the string template, enabling dynamic input for context and question
prompt = ChatPromptTemplate.from_template(prompt_str)

# Create a retrieval pipeline to fetch relevant context and pass through the user's question
retrieval = RunnableParallel(
    {
        # Use the vector database as a retriever to fetch relevant context for the question
        "context": vectordb.as_retriever(),

        # Pass through the user's input question without modification
        "question": RunnablePassthrough()
    }
)

# Define an output parser to format the generated response into a string
output_parser = StrOutputParser()

# Create a processing chain that retrieves context, formats the prompt, generates an LLM response, and parses the output
chain = retrieval | prompt | llm | output_parser

In [11]:
# Invoke the chain of components (retrieval, prompt generation, LLM processing, and output parsing)
# The question "What is YOLO?" is passed through the chain to generate the response
response = chain.invoke("What is YOLO?")

# Print the response generated by the chain
print(response)

YOLO (You Only Look Once) is an object detection system that uses a single convolutional neural network to detect objects in an image. It is a fast and accurate detector, making it suitable for real-time applications.

YOLO has several advantages over traditional object detection methods:
- Speed: YOLO frames detection as a regression problem, allowing it to process images quickly. It can run at 45 frames per second on a Titan X GPU, and a faster version can achieve over 150 fps, making it suitable for real-time video processing with low latency.
- Global reasoning: Unlike sliding window or region proposal-based techniques, YOLO considers the entire image during training and testing, allowing it to encode contextual information about classes and their appearance. This helps it make fewer background errors compared to methods like Fast R-CNN.
- Generalizability: YOLO learns generalizable representations of objects, allowing it to perform well when trained on natural images and tested on

## Other chain invoking methods!

.invoke(): The goal is to pass in an input and receive the output—neither more nor less.

.batch(): This is faster than using invoke three times when you wish to supply several inputs to get multiple outputs because it handles the parallelization for you.

.stream():  We may begin printing the response before the entire response is complete.

In [12]:
response_with_batch = chain.batch(["What is Transformers", "How is Transformer different than YOLO?"])

for response in response_with_batch:
  print(response)
  print("\n")

The Transformer is a model architecture that aims to reduce sequential computation in sequence modeling and transduction problems, such as language modeling and machine translation. It replaces the traditional recurrent layers in encoder-decoder architectures with multi-headed self-attention, allowing for more parallelization and faster training.

The Transformer is the first sequence transduction model based entirely on attention, and it has achieved state-of-the-art performance in various translation tasks, outperforming even some ensemble models. It computes global dependencies between input and output, and its self-attention mechanism relates different positions of a single sequence to generate a representation of the sequence.

The Transformer's architecture consists of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder maps an input sequence to a continuous representation, and the decoder generates an output sequence based 

In [13]:
for chunk in chain.stream("What are the 3 vectors in Transformers architecture?"):
  print(chunk, flush=True, end="")

The three vectors in the Transformer architecture are:

1. Query (q): This vector represents the current position in the input sequence and is used to calculate the attention scores.
2. Key (k): The key vector is used to calculate the attention scores and helps to determine the relevance of different parts of the input sequence to the query.
3. Value (v): This vector contains the information from the input sequence that the model attends to. It is weighted by the attention scores and then summed up to get the final output.

These vectors are used in the self-attention mechanism, which is a core component of the Transformer model, allowing it to capture global dependencies between input and output.